### ARC Challenge Exploration Notebook


### 1. Setup and Imports


In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from typing import Dict, List, Tuple

# Append the parent directory to the system path to import local modules
sys.path.append('..')

# Import custom modules
from config import Config
from data.data_loader import ARCDataLoader
from data.data_preprocessor import ARCPreprocessor
from visualization.grid_viz import GridVisualizer
from utils.helpers import (
    analyze_grid_patterns,
    find_transformations,
    detect_symmetry,
    calculate_grid_statistics
)

### 2. Data Loading and Basic Statistics

In [ ]:
# Initialize the ARC data loader
loader = ARCDataLoader(Config.TRAINING_DIR)
tasks = loader.load_all_tasks()

# Print the number of tasks loaded
print(f"Total training tasks: {len(tasks)}")

# Initialize lists to collect grid statistics
grid_sizes = []
color_counts = []
transformation_types = []

# Iterate through tasks to gather statistics
for task in tasks:
    train_pairs, _ = loader.get_train_test_pairs(task)
    for input_grid, output_grid in train_pairs:
        grid_sizes.append(input_grid.shape)
        color_counts.append(len(np.unique(input_grid)))
        transformation_types.append(find_transformations(input_grid, output_grid))

#### Visualization of Basic Grid Statistics

In [ ]:
plt.figure(figsize=(15, 5))

# Plot grid heights
plt.subplot(131)
plt.hist([s[0] for s in grid_sizes], bins=30)
plt.title('Grid Heights')
plt.xlabel('Height')

# Plot grid widths
plt.subplot(132)
plt.hist([s[1] for s in grid_sizes], bins=30)
plt.title('Grid Widths')
plt.xlabel('Width')

# Plot unique color counts per grid
plt.subplot(133)
plt.hist(color_counts, bins=10)
plt.title('Unique Colors per Grid')
plt.xlabel('Number of Colors')

plt.tight_layout()
plt.show()

### 3. Analyzing Patterns in Tasks

In [ ]:
def analyze_task_patterns(task):
    """
    Analyzes and visualizes the input-output pairs of a given task.
    """
    train_pairs, test_pairs = loader.get_train_test_pairs(task)
    
    print("Training Pairs Analysis:")
    for idx, (input_grid, output_grid) in enumerate(train_pairs):
        print(f"\nPair {idx + 1}:")

        # Visualize input-output grid pair
        GridVisualizer.visualize_sample(input_grid, output_grid)

        # Analyze and print detected patterns in the input grid
        patterns = analyze_grid_patterns(input_grid)
        print("\nInput Grid Patterns:")
        for pattern_type, details in patterns.items():
            print(f"{pattern_type}: {details}")

        # Analyze and print symmetry in the input grid
        symmetry = detect_symmetry(input_grid)
        print("\nSymmetry Analysis:")
        print(symmetry)

        # Calculate and print statistics of the grid
        stats = calculate_grid_statistics(input_grid, output_grid)
        print("\nGrid Statistics:")
        for stat_name, value in stats.items():
            print(f"{stat_name}: {value}")

#### Analyze a few sample tasks

In [ ]:

def visualize_transformations(input_grid, output_grid):
    """
    Visualizes the transformation process from the input grid to the output grid
    by applying detected transformations step by step.
    """
    # Identify basic transformations between the input and output grids
    transformations = find_transformations(input_grid, output_grid)
    
    # Initialize a list to store transformation steps
    steps = [input_grid]
    current_grid = input_grid.copy()
    
    # Apply each transformation sequentially and store the intermediate results
    for trans_type, params in transformations.items():
        current_grid = apply_transformation(current_grid, trans_type, params)
        steps.append(current_grid)
    
    # Visualize each step
    n_steps = len(steps)
    fig, axes = plt.subplots(1, n_steps, figsize=(5 * n_steps, 5))
    
    for i, step in enumerate(steps):
        if n_steps == 1:
            ax = axes
        else:
            ax = axes[i]
        ax.imshow(step, cmap='tab10')
        ax.grid(True)
        if i == 0:
            ax.set_title('Input')
        elif i == n_steps - 1:
            ax.set_title('Output')
        else:
            ax.set_title(f'Step {i}')
    
    plt.tight_layout()
    plt.show()

### Apply the transformation visualization to the sample tasks

In [ ]:
for task in sample_tasks:
    train_pairs, _ = loader.get_train_test_pairs(task)
    for input_grid, output_grid in train_pairs:
        visualize_transformations(input_grid, output_grid)